# Benchmarking supervised machine learning classifiers for transcriptome-based *Salmonella* resistance prediction in chickens

Analysis notebook. Data: NCBI GEO accession **GSE168060** (Dar et al., 2022, *Genomics* 114(5)), supplementary file `GSE168060_gene_exp.txt`.

**How to run (Google Colab):** run the cells from top to bottom. When asked, upload `GSE168060_gene_exp.txt`.

**Design in brief**
- 11 infected samples (5 resistant Kashmir Faverolla, 6 susceptible broiler; liver and spleen pooled); uninfected controls are removed.
- Labels are assigned by breed, not by individually measured resistance.
- Pipeline fitted inside every cross-validation fold: standardisation, variance filter, ANOVA F-test selection of 500 genes, classifier.
- Hyperparameters were fixed in advance and not tuned. Random seed: 42 everywhere.

## 1. Setup

In [ ]:
%pip install -q xgboost

import os
os.environ["PYTHONWARNINGS"] = "ignore"   # keep the output readable (also applies to parallel workers)
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42

def download(path):
    """Download a file when running in Google Colab; do nothing elsewhere."""
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        pass

## 2. Load and clean the expression data

In [ ]:
RAW_FILE = "GSE168060_gene_exp.txt"   # GEO supplementary file for GSE168060

if not os.path.exists(RAW_FILE):
    try:
        from google.colab import files
        print(f"Please upload {RAW_FILE}")
        files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {RAW_FILE} in the working directory.")
print("Found:", RAW_FILE)

In [ ]:
# The raw file contains a malformed line (wrong number of tab-separated fields).
# Such lines are padded or truncated to the header width, and reported below.
CLEAN_FILE = "GSE168060_gene_exp_clean.txt"
bad_lines = []

with open(RAW_FILE, "r", encoding="utf-8", errors="replace") as fin, \
     open(CLEAN_FILE, "w", encoding="utf-8") as fout:
    header = fin.readline()
    fout.write(header)
    n_cols = len(header.rstrip("\n").split("\t"))
    for line_no, line in enumerate(fin, start=2):
        parts = line.rstrip("\n").split("\t")
        if len(parts) != n_cols:
            bad_lines.append((line_no, len(parts), parts[0]))
            parts = (parts + [""] * n_cols)[:n_cols]
        fout.write("\t".join(parts) + "\n")

print("Malformed lines repaired:", len(bad_lines))
for line_no, n, gene in bad_lines:
    print(f"  line {line_no}: {n} fields (expected {n_cols}); first field = {gene!r}")

In [ ]:
df_raw = pd.read_csv(CLEAN_FILE, sep="\t")
print("Raw shape (genes x columns):", df_raw.shape)

# samples x genes
df = df_raw.set_index(df_raw.columns[0]).transpose()
print("After transpose (samples x genes):", df.shape)

# remove uninfected controls (sample names containing "C")
df = df[~df.index.str.contains("C")]
print("After removing controls:", df.shape)
print("Samples:", list(df.index))

In [ ]:
# Labels by breed: B = commercial broiler (susceptible), F = Kashmir Faverolla (resistant)
labels = []
for sample in df.index:
    if sample.startswith("B"):
        labels.append("Susceptible")
    elif sample.startswith("F"):
        labels.append("Resistant")
    else:
        labels.append(None)

df["Label"] = labels
assert df["Label"].isna().sum() == 0, "Some samples could not be labelled - check sample names."

X = df.drop(columns=["Label"])
y = df["Label"]
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

# Explicit encoding: Resistant = 1 (positive class), Susceptible = 0
y_encoded = (y == "Resistant").astype(int).values

print("X shape:", X.shape)
print(y.value_counts())
print("Missing values in X:", X.isnull().sum().sum())

## 3. Pipelines and classifiers

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

VAR_THRESH = 0.0
K_BEST = 500

models = {
    "RF": RandomForestClassifier(n_estimators=500, random_state=SEED),
    "SVM-RBF": SVC(kernel="rbf", probability=True, random_state=SEED),
    "XGBoost": XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=3,
        subsample=0.9, colsample_bytree=0.9,
        random_state=SEED, eval_metric="logloss"
    ),
    "NN-MLP": MLPClassifier(hidden_layer_sizes=(50, 20), max_iter=3000, random_state=SEED),
    "NaiveBayes": GaussianNB(),
}

pipelines = {
    name: Pipeline([
        ("scaler", StandardScaler()),
        ("var_filter", VarianceThreshold(threshold=VAR_THRESH)),
        ("select_k", SelectKBest(score_func=f_classif, k=min(K_BEST, X.shape[1]))),
        ("model", model),
    ])
    for name, model in models.items()
}

## 4. Benchmark: repeated stratified 5-fold cross-validation (30 repeats)

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, matthews_corrcoef

cv_table = RepeatedStratifiedKFold(n_splits=5, n_repeats=30, random_state=SEED)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
}

results = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y_encoded, cv=cv_table, scoring=scoring, n_jobs=-1)
    results.append({
        "Model": name,
        "Accuracy (mean)": np.mean(scores["test_accuracy"]),
        "Balanced Acc (mean)": np.mean(scores["test_balanced_accuracy"]),
        "F1 (mean)": np.mean(scores["test_f1"]),
        "ROC-AUC (mean)": np.mean(scores["test_roc_auc"]),
        "MCC (mean)": np.mean(scores["test_mcc"]),
    })

results_df = pd.DataFrame(results).sort_values(by="MCC (mean)", ascending=False)
results_df.to_csv("benchmark_results.csv", index=False, float_format="%.2f")
results_df.round(2)

## 5. ROC curves and confusion matrices (one shuffled stratified 5-fold split)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

cv_single = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

full_names = {"RF": "Random Forest", "SVM-RBF": "SVM - RBF Kernel", "XGBoost": "XGBoost",
              "NN-MLP": "Neural Network (MLP)", "NaiveBayes": "Naive Bayes"}
colors = {"RF": "#2196F3", "SVM-RBF": "#FF9800", "XGBoost": "#4CAF50",
          "NN-MLP": "#F44336", "NaiveBayes": "#9C27B0"}

# Out-of-fold predictions, computed once and reused for both figures.
# Column 1 of predict_proba is class 1 = Resistant.
oof_proba, oof_pred = {}, {}
for name, pipe in pipelines.items():
    oof_proba[name] = cross_val_predict(pipe, X, y_encoded, cv=cv_single,
                                        method="predict_proba", n_jobs=-1)[:, 1]
    oof_pred[name] = cross_val_predict(pipe, X, y_encoded, cv=cv_single, n_jobs=-1)

# --- ROC curves ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, name in enumerate(pipelines):
    fpr, tpr, _ = roc_curve(y_encoded, oof_proba[name])
    ax = axes[i]
    ax.fill_between(fpr, tpr, alpha=0.08, color=colors[name])
    ax.plot(fpr, tpr, color=colors[name], linewidth=3.5)
    ax.plot([0, 1], [0, 1], color="#aaaaaa", linestyle="--", linewidth=1)
    ax.text(0.97, 0.08, f"AUC = {auc(fpr, tpr):.3f}", transform=ax.transAxes, ha="right",
            fontsize=11, fontweight="bold", color=colors[name],
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=colors[name]))
    ax.set_title(full_names[name], fontsize=12, fontweight="bold")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.2, linestyle="--")
axes[5].set_visible(False)
fig.suptitle("ROC Curves - Salmonella Resistance Classification\n(StratifiedKFold, 5 Folds)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout(pad=2.5)
fig.savefig("roc_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# --- Confusion matrices (rows/columns ordered 0 = Susceptible, 1 = Resistant) ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, name in enumerate(pipelines):
    cm = confusion_matrix(y_encoded, oof_pred[name], labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Susceptible", "Resistant"])
    disp.plot(ax=axes[i], colorbar=False, cmap=plt.cm.Blues)
    for t in axes[i].texts:
        t.set_fontsize(16); t.set_fontweight("bold")
    axes[i].set_title(full_names[name], fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Predicted Label"); axes[i].set_ylabel("True Label")
axes[5].set_visible(False)
fig.suptitle("Confusion Matrices - Salmonella Resistance Classification\n(StratifiedKFold, 5 Folds)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout(pad=2.5)
fig.savefig("confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. PCA of the 11 infected samples (standardised expression values, all genes)

In [ ]:
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
var = pca.explained_variance_ratio_ * 100
print(f"PC1 {var[0]:.1f}%, PC2 {var[1]:.1f}%")

fig, ax = plt.subplots(figsize=(8, 6))
for name, color in [("Susceptible", "#1f77b4"), ("Resistant", "#ff7f0e")]:
    mask = (y == name).values
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, s=100)
ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title("PCA of Transcriptomic Profiles")
ax.legend(); ax.grid(True)
fig.savefig("pca_plot.png", dpi=300, bbox_inches="tight")   # save before show
plt.show()

## 7. Random Forest feature importance

The pipeline is fitted on **all 11 samples**, so these scores are descriptive and were not cross-validated.

In [ ]:
rf_pipe = pipelines["RF"]
rf_pipe.fit(X, y_encoded)

# Apply BOTH selection steps (variance filter, then SelectKBest) to recover gene names
var_mask = rf_pipe.named_steps["var_filter"].get_support()
k_mask = rf_pipe.named_steps["select_k"].get_support()
selected_genes = X.columns[var_mask][k_mask]

importances = rf_pipe.named_steps["model"].feature_importances_
feat_df = (pd.DataFrame({"Gene": selected_genes, "Importance": importances})
           .sort_values("Importance", ascending=False)
           .head(50).reset_index(drop=True))
feat_df.to_csv("top50_genes_rf_importance.csv", index=False)
feat_df.head(10)

In [ ]:
from matplotlib.patches import Patch

top_genes = feat_df["Gene"].tolist()
res_mask = (y == "Resistant").values
sus_mask = (y == "Susceptible").values
res_mean = X.loc[res_mask, top_genes].mean()
sus_mean = X.loc[sus_mask, top_genes].mean()

bar_colors = ["#F44336" if res_mean[g] > sus_mean[g] else "#2196F3" for g in top_genes]

fig, ax = plt.subplots(figsize=(10, 16))
bars = ax.barh(feat_df["Gene"][::-1], feat_df["Importance"][::-1],
               color=bar_colors[::-1], edgecolor="white", height=0.75)
for bar, val in zip(bars, feat_df["Importance"][::-1]):
    ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", ha="left", fontsize=7, color="#555555")
ax.legend(handles=[Patch(facecolor="#F44336", label="Higher in Resistant"),
                   Patch(facecolor="#2196F3", label="Higher in Susceptible")],
          loc="lower right", fontsize=10)
ax.set_xlabel("Feature Importance Score"); ax.set_ylabel("Gene")
ax.set_title("Top 50 Predictive Genes - Random Forest Feature Importance\n(Colored by Higher Expression Class)",
             fontsize=13, fontweight="bold", pad=15)
ax.tick_params(labelsize=8)
ax.grid(True, axis="x", alpha=0.2, linestyle="--")
plt.tight_layout()
fig.savefig("top50_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

heat = X[top_genes].T
heat.columns = [f"{s} ({'R' if lab == 'Resistant' else 'S'})" for s, lab in zip(X.index, y)]

fig, ax = plt.subplots(figsize=(12, 16))
sns.heatmap(heat, ax=ax, cmap="Reds", linewidths=0.3, linecolor="#eeeeee",
            xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": 0.5, "label": "Expression Level"})
ax.set_title("Top 50 Predictive Genes - Expression Heatmap\n(R = Resistant [Faverolla], S = Susceptible [broiler])",
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Sample"); ax.set_ylabel("Gene")
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
fig.savefig("top50_genes_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Software versions and downloads

In [ ]:
import sys, sklearn, xgboost, seaborn
print("Python", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("xgboost", xgboost.__version__)
print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", seaborn.__version__)

In [ ]:
outputs = ["benchmark_results.csv", "top50_genes_rf_importance.csv", "roc_curves.png",
           "confusion_matrices.png", "pca_plot.png", "top50_feature_importance.png",
           "top50_genes_heatmap.png"]
import zipfile
with zipfile.ZipFile("results.zip", "w") as z:
    for f in outputs:
        if os.path.exists(f):
            z.write(f)
print("Saved results.zip")
download("results.zip")